In [ ]:
# 
# Import packages
#
import pandas as pd
import numpy as np
import os
import pickle
import json

In [ ]:
#
# Import shared functions and schemas from shared.py
#
from shared import (
    get_embedding,
    cosine_similarity,
    rank_course_matches,
    load_course_embeddings,
    combine_unique_matches,
    format_matches_for_gpt,
    transfer_match_schema,
    review_top_matches_with_gpt,
    evaluate_transfer_course,
    course_lookup_schema,
    find_course_description_online,
)

In [2]:
# 
# Load the file with UR course information
# 
raw_df = pd.read_csv("Fall2026Registration.csv")

In [3]:
# 
# Clean file to remove duplicates 
# 
course_df = raw_df[
    ["SUBJ", "CRSE", "TITLE", "COURSE_TEXT", "COURSE_DESC"]
].copy()

course_df["SUBJ"] = course_df["SUBJ"].fillna("").astype(str).str.strip()
course_df["CRSE"] = course_df["CRSE"].fillna("").astype(str).str.strip()
course_df["TITLE"] = course_df["TITLE"].fillna("").astype(str).str.strip()
course_df["COURSE_TEXT"] = course_df["COURSE_TEXT"].fillna("").astype(str).str.strip()
course_df["COURSE_DESC"] = course_df["COURSE_DESC"].fillna("").astype(str).str.strip()

course_df = course_df.drop_duplicates(
    subset=["SUBJ", "CRSE", "TITLE"],
    keep="first"
)

In [4]:
# 
# Save to Excel file to use with ease 
# 
course_df.to_excel(
    "clean_ur_courses.xlsx",
    index=False
)

In [ ]:
#
# Creating two embedding indexes for UR courses (Run once - otherwise run the load_course_embeddings function)
#
# Index A: title + description (content-based retrieval)
#
# Skip if created embeddings already
#

title_description_embeddings = []
all_details_embeddings = []

for _, row in course_df.iterrows():

    title_desc_text = (
        str(row["TITLE"]) +
        "\n\n" +
        str(row["COURSE_DESC"])
    )

    all_details_text = (
        str(row["SUBJ"]) + " " + str(row["CRSE"]) +
        "\n\n" +
        str(row["TITLE"]) +
        "\n\n" +
        str(row["COURSE_DESC"]) +
        "\n\n" +
        str(row["COURSE_TEXT"])
    )

    title_desc_embedding = get_embedding(title_desc_text)
    all_details_embedding = get_embedding(all_details_text)

    course_record = {
        "subject": row["SUBJ"],
        "course_number": row["CRSE"],
        "course_title": row["TITLE"],
        "course_description": row["COURSE_DESC"],
    }

    title_description_embeddings.append({
        **course_record,
        "embedding": title_desc_embedding
    })

    all_details_embeddings.append({
        **course_record,
        "embedding": all_details_embedding
    })

len(title_description_embeddings), len(all_details_embeddings)

In [ ]:
#
# Save both embedding indexes to pickle files for easy retrieval
#
# Skip if created embeddings already
#

os.makedirs("data", exist_ok=True)

with open("data/ur_title_description_embeddings.pkl", "wb") as file:
    pickle.dump(title_description_embeddings, file)

with open("data/ur_all_details_embeddings.pkl", "wb") as file:
    pickle.dump(all_details_embeddings, file)

In [9]:
#
# Load both course embedding indexes without regenerating embeddings (Run when embeddings have already been created)
# Run everytime 
#

# Run this cell only when reopening the notebook and you do not want to regenerate embeddings.
title_description_embeddings, all_details_embeddings = load_course_embeddings()

In [37]:
#
# Prompts user for outside course details to apply the match to 
#
outside_university = input("Enter university name: ")
outside_subject = input("Enter course subject (e.g. ECON, BIOL, MATH): ")
outside_code = input("Enter course code (e.g. 101, 205, 320): ")
outside_title = input("Enter course title: ")
outside_term = input("Enter term (optional, press Enter to skip): ").strip() or None
outside_year = input("Enter year (optional, press Enter to skip): ").strip() or None

In [38]:
#
# Call web search, display retrieved info for verification, allow description edit, require approval
#
lookup_result = find_course_description_online(
    university=outside_university,
    subject=outside_subject,
    code=outside_code,
    title=outside_title,
    term=outside_term,
    year=outside_year
)

print("Retrieved course information:")
print(f"University: {lookup_result['university']}")
print(f"Subject:    {lookup_result['subject']}")
print(f"Code:       {lookup_result['code']}")
print(f"Title:      {lookup_result['title']}")
print(f"Source URL: {lookup_result['source_url'] if lookup_result['found'] else '(not found)'}")
print()

if lookup_result["found"]:
    print("Description (found online):")
    print(lookup_result["description"])
else:
    print("No description was found online. Please enter one manually.")

print()
edited_description = input(
    "Press Enter to accept the description above as-is, or type a replacement description: "
)

if edited_description.strip():
    outside_description = edited_description.strip()
else:
    outside_description = lookup_result["description"]

approve = input("Approve and Continue? (Y/n): ")

if approve.strip().lower() not in ("y", "yes", ""):
    raise ValueError("Course information not approved. Re-run this cell to try again.")

print()
print("Approved. Final course description:")
print(outside_description)

Retrieved course information:
University: Georgetown
Subject:    ECON
Code:       1001
Title:      Principles of Microeconomics
Source URL: https://summersessions.georgetown.edu/programs/362/summer-school-undergraduatecourses/

Description (found online):
This course first develops simple graphical and mathematical models of decision-making by individual economic agents: consumers, workers, and businesses. We analyze interactions between these agents in product and factor markets using concepts of market demand, supply, and equilibrium. Finally, we demonstrate the efficiency of perfectly competitive markets, describe the conditions under which that efficiency arises, and examine market failures that occur when those conditions are not met.


Approved. Final course description:
This course first develops simple graphical and mathematical models of decision-making by individual economic agents: consumers, workers, and businesses. We analyze interactions between these agents in product an

In [29]:
#
# Run the full pipeline: two retrieval passes, merge, and LLM review -> top 5 matches
#
candidate_pool, review_df = evaluate_transfer_course(
    outside_university=outside_university,
    outside_subject=outside_subject,
    outside_code=outside_code,
    outside_title=outside_title,
    outside_description=outside_description,
    title_description_embeddings=title_description_embeddings,
    all_details_embeddings=all_details_embeddings,
    top_n=10
)

candidate_pool, review_df

(   subject course_number                    course_title  \
 0     CMSC           323  DSGN/IMPLEMNTN PROG LANG W/LAB   
 1     CMSC           150   INTRODUCTN TO COMPUTING W/LAB   
 2     DSST           289    INTRODUCTION TO DATA SCIENCE   
 3     ECON           242       DATA ANALYSIS & COMPUTING   
 4     CMSC           301     COMPUTER ORGANIZATION W/LAB   
 5     CMSC           221           DATA STRUCTURES W/LAB   
 6     MATH           300   FUNDAMENTALS OF ABSTRACT MATH   
 7     BIOL           351     ST: APPL SCI COMP IN PYTHON   
 8     CMSC           325         DATABASE SYSTEMS W/ LAB   
 9     CMSC           240  SOFTWARE SYSTEMS DVLPMNT W/LAB   
 10    CMSC           322    SOFTWARE ENGNRNG PRACT W/LAB   
 11    CMSC           101           AI AND THE HUMAN MIND   
 12    CMSC           315               ALGORITHMS W/ LAB   
 
                                    course_description  score_title_desc  \
 0   Concepts in design and implementation of progr...          0.50

In [30]:
#
# View merged candidate pool (from both retrieval passes)
#
candidate_pool

,subject,course_number,course_title,course_description,score_title_desc,score_all_details
0,CMSC,323,DSGN/IMPLEMNTN PROG LANG W/LAB,Concepts in design and implementation of progr...,0.508501,0.537493
1,CMSC,150,INTRODUCTN TO COMPUTING W/LAB,Techniques for writing computer programs to so...,0.499840,0.555093
2,DSST,289,INTRODUCTION TO DATA SCIENCE,"Topics will include techniques for collecting,...",0.492392,0.514139
3,ECON,242,DATA ANALYSIS & COMPUTING,Introduction to data analysis and programming ...,0.482115,0.534565
4,CMSC,301,COMPUTER ORGANIZATION W/LAB,Fundamentals of computer organization. Topics ...,0.478685,0.524667
5,CMSC,221,DATA STRUCTURES W/LAB,"Introduction to data structures, including sta...",0.452395,0.510297
6,MATH,300,FUNDAMENTALS OF ABSTRACT MATH,"Logic, quantifiers, negations of statements wi...",0.449306,NaN
7,BIOL,351,ST: APPL SCI COMP IN PYTHON,"Cap is 10 for 4th yrs, 16 for 3rd. To request ...",0.435986,NaN
8,CMSC,325,DATABASE SYSTEMS W/ LAB,Introduction to systematic management of data:...,0.435985,NaN
9,CMSC,240,SOFTWARE SYSTEMS DVLPMNT W/LAB,Introduction to techniques necessary for devel...,0.435787,0.491428


In [31]:
#
# View top 5 matches with reasoning 
#
review_df

,rank,course_code,course_title,subject_alignment,content_overlap,course_level_alignment,uncertainties,reason,confidence
0,1,CMSC 150 - INTRODUCTN TO COMPUTING W/LAB,INTRODUCTN TO COMPUTING W/LAB,High (Computer Science to Computer Science),"High (intro programming concepts, OOP, control...",Introductory alignment (beginner level),UR may use languages other than Python in this...,Excellent match to the student’s stated need f...,High
1,2,CMSC 221 - DATA STRUCTURES W/LAB,DATA STRUCTURES W/LAB,High (Computer Science to Computer Science),"High (data structures, recursion, abstraction,...",Intermediate mismatch (200-level vs introducto...,Topic emphasis on data structures may extend b...,Strong overlap on core CS concepts that typica...,Medium
2,3,CMSC 240 - SOFTWARE SYSTEMS DVLPMNT W/LAB,SOFTWARE SYSTEMS DVLPMNT W/LAB,High (Computer Science to Computer Science),"High (software development lifecycle, OOP desi...",Advanced mismatch (300-level),Language specifics (C++ and STL) and project s...,Covers essential software engineering practice...,Low
3,4,CMSC 322 - SOFTWARE ENGNRNG PRACT W/LAB,SOFTWARE ENGNRNG PRACT W/LAB,High (Computer Science to Computer Science),"Medium-High (software engineering practices, p...",Advanced mismatch (300-level),Project emphasis and advanced design topics ma...,Useful for understanding engineering practices...,Medium
4,5,CMSC 323 - DSGN/IMPLEMNTN PROG LANG W/LAB,DSGN/IMPLEMNTN PROG LANG W/LAB,High (Computer Science to Computer Science),Medium (object-oriented concepts and language ...,Advanced mismatch (300-level),Focus on programming language design and imple...,Offers relevant exposure to design and impleme...,Low
